Each disk is recorded once a day, so each daily CSV contains different snapshots. Merging all of them together would produce too many instances to train on efficiently.

In [ ]:
import pandas as pd

# Quick sanity check: compare disk snapshots across two consecutive days
df_07_01 = pd.read_csv('data/Q3/2025-07-01.csv', sep=';')
df_07_02 = pd.read_csv('data/Q3/2025-07-02.csv', sep=';')

display(df_07_01.head())
display(df_07_02.head())

We collect 4414 failure=1 instances from the full year of data. The other 4414 will be selected randomly from healthy disks across all files, so the training set is well distributed.

In [ ]:
import pandas as pd
import glob
import os

all_files = glob.glob(os.path.join('data/**', "*.csv"), recursive=True)

failed_disks_list = []
total_failures = 0

for i, file_path in enumerate(all_files):
    try:
        temp_df = pd.read_csv(file_path, low_memory=False)

        failures = temp_df[temp_df['failure'] == 1].copy()

        if not failures.empty:
            failed_disks_list.append(failures)
            total_failures += len(failures)

        # Progress update every 50 files so you know something is happening
        if i % 50 == 0:
            print(f"Processed {i}/{len(all_files)} files... Failures found so far: {total_failures}")

    except Exception as e:
        print(f"Error reading {os.path.basename(file_path)}: {e}")

if failed_disks_list:
    df_all_failures = pd.concat(failed_disks_list, ignore_index=True)
    print("\n" + "="*40)
    print(f"FINAL RESULT FOR THE FULL YEAR:")
    print(f"Total failures found (failure=1): {total_failures}")
    print("="*40)

    df_all_failures.to_csv('all_failures.csv', index=False)
    print("All failures saved to 'all_failures.csv'.")

    display(df_all_failures.head())
else:
    print("No failures were found across any of the data files.")

The other half of the training set consists of healthy disks (failure=0). We select them randomly and cap each file at 20 instances to ensure the sample is spread across the full year rather than concentrated in a few dates.

In [ ]:
import pandas as pd
import glob
import os
import random

all_files = glob.glob(os.path.join('data/', "**/*.csv"), recursive=True)
target_count = 4414

healthy_disks_list = []
collected_healthy = 0

# Shuffle file list so healthy samples are spread across the full year
random.shuffle(all_files)

for file_path in all_files:
    if collected_healthy >= target_count:
        break

    try:
        temp_df = pd.read_csv(file_path, low_memory=False)
        healthy = temp_df[temp_df['failure'] == 0]

        if not healthy.empty:
            # Cap at 20 instances per file to avoid any single date dominating the sample
            sample_size = min(len(healthy), 20)
            sample = healthy.sample(n=sample_size)

            healthy_disks_list.append(sample)
            collected_healthy += len(sample)

            if collected_healthy % 500 < 20:  # Progress update
                print(f"Collected: {collected_healthy}/{target_count} healthy instances...")

    except Exception as e:
        continue

df_healthy_final = pd.concat(healthy_disks_list, ignore_index=True).head(target_count)

# Combine failed and healthy disks into one balanced dataset
df_model_ready = pd.concat([df_all_failures, df_healthy_final], ignore_index=True)

# Shuffle rows so failure=1 and failure=0 instances are interleaved randomly
df_model_ready = df_model_ready.sample(frac=1).reset_index(drop=True)

print(f"Final dataset (failure=1 + failure=0)... Total rows: {len(df_model_ready)}")

# Save the final balanced dataset
df_model_ready.to_csv('model_ready_data.csv', index=False)